# Airport Cargo MAWM：Colab 快速闭环

这个 Notebook 会完成：安装 → 环境冒烟测试 → Oracle CEM → 采集小数据 → 训练状态世界模型 → 学习模型 CEM 评估。

这是状态空间 MVP，不是最终的像素 JEPA。正式实验前请先确保这里的每一格都能运行。

## 1. 克隆或更新代码

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/june250630/airport-cargo-mawm.git"
REPO_DIR = Path("/content/airport-cargo-mawm")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("工作目录：", Path.cwd())

## 2. 安装依赖并检查 GPU

Colab 菜单中选择“代码执行程序 → 更改运行时类型 → GPU”。状态基线很小，CPU 也能训练；后续像素模型才真正需要 GPU。

In [ ]:
!python -m pip install -q -e ".[dev]"
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. 运行单元测试

In [ ]:
!pytest

## 4. 脚本专家冒烟测试

脚本专家读取真实仿真状态，只用于证明任务可解以及采集训练数据。

In [ ]:
LOCAL_ROOT = Path("/content/acmawm_runs")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
!python -m airport_cargo_mawm.cli.smoke --episodes 3 --videos 1 --output /content/acmawm_runs/smoke
from IPython.display import Video, display
display(Video("/content/acmawm_runs/smoke/episode_000.mp4", embed=True))

## 5. 真实动力学 Oracle CEM

Oracle使用特权全局状态和精确动力学，只用于判断CEM搜索是否能完成任务，不是可部署策略或数学最优上界。

In [ ]:
!python -m airport_cargo_mawm.cli.oracle --cost task --episodes 2 --seed 0 --search-seed 1000 --horizon 20 --samples 256 --elites 32 --iterations 4 --videos 1 --output /content/acmawm_runs/oracle_task
display(Video("/content/acmawm_runs/oracle_task/episode_000.mp4", embed=True))

## 6. 采集小数据并训练 5 个 epoch

这只是管线测试。正式实验应扩大场景数、使用独立保留集，并训练多个随机种子。

In [ ]:
!python -m airport_cargo_mawm.cli.collect --episodes 20 --seed 0 --output /content/acmawm_runs/train.h5
!python -m airport_cargo_mawm.cli.train --data /content/acmawm_runs/train.h5 --output /content/acmawm_runs/checkpoint --epochs 5 --batch-size 128 --seed 0

## 7. 用学习模型进行小规模 CEM 评估

初始模型可能无法成功完成任务；这里首先验证规划代码能否执行并生成结果。

In [ ]:
!python -m airport_cargo_mawm.cli.evaluate --checkpoint /content/acmawm_runs/checkpoint/last.pt --cost task --episodes 2 --videos 1 --samples 64 --elites 8 --iterations 2 --output /content/acmawm_runs/eval
display(Video("/content/acmawm_runs/eval/episode_000.mp4", embed=True))

## 8. 持久化到 Google Drive

`/content` 会在运行时结束后被删除。确认前面全部成功后，再运行本格保存数据和权重。

In [ ]:
from google.colab import drive
import shutil
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/airport_cargo_mawm/quickstart")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_ROOT, DRIVE_ROOT, dirs_exist_ok=True)
print("已保存到：", DRIVE_ROOT)

## 下一步

1. 把数据扩展到至少 100 个训练场景和独立测试场景；
2. 训练 3 个不同随机种子的模型，组成模型集成；
3. 对比 `uncertainty_weight=0` 与大于 0 的校准规划；
4. 对比 Oracle 与学习模型在相同 task/shaped 代价下的差距；
5. 扩展到 VMAS 机场场景和像素 JEPA。